# 02 — Cleaning and SQL Analytics

This notebook runs the repository's tested ingestion/cleaning implementation and then executes the DuckDB business queries. It preserves cancellation rows for returns analysis and flags valid purchases separately.

**Prerequisite:** Complete Notebook 01 and place the official workbook at `data/raw/online_retail_II.xlsx`.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src" / "data.py").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.data import main as ingest_data, PARQUET_FILE, DB_FILE
from src.run_sql import main as run_analytics

print("Project:", PROJECT_ROOT)

## 1. Ingest, clean and persist

The pipeline standardizes column names, parses dates and numeric fields, removes exact duplicate rows, flags cancellation invoices and defines a valid purchase as a non-cancellation row with positive quantity and unit price. Rows without Customer ID remain available for transaction-level analysis.

In [ ]:
ingest_data()
print("Parquet created:", PARQUET_FILE.exists())
print("DuckDB created:", DB_FILE.exists())

## 2. Inspect cleaned transaction flags

In [ ]:
clean = pd.read_parquet(PARQUET_FILE)
print("Cleaned rows:", len(clean))
display(clean.head())
display(clean.groupby(["is_cancellation", "is_valid_purchase"], dropna=False).size().rename("rows").to_frame())
print("Valid purchase rows:", int(clean["is_valid_purchase"].sum()))

## 3. Run business SQL

The SQL script exports monthly revenue, country performance, cancellations, RFM segments, cohort retention and customer revenue concentration. These are descriptive analytics, not model performance results.

In [ ]:
run_analytics()
SQL_OUTPUT = PROJECT_ROOT / "reports" / "sql"
print("Exported files:", sorted(p.name for p in SQL_OUTPUT.glob("*.csv")))

## 4. Review the outputs

In [ ]:
for filename in ["monthly_revenue.csv", "country_revenue.csv", "cancellations.csv", "rfm_segments.csv"]:
    path = SQL_OUTPUT / filename
    if path.exists():
        print(f"\\n{filename}")
        display(pd.read_csv(path).head(10))

## Interpretation prompts

- Is revenue concentrated in a small number of countries or customers?
- Do cancellation values vary over time?
- Which RFM segments have the largest customer counts and average revenue?
- How does cohort retention change as month number increases?

Do not infer campaign impact from these descriptive patterns.